###Remove all where claim_id, policy_id is ,claim status,claim_amount,lastupdated null
###Remove all rows where policy_id Id not exist in Policy table
###Convert date_of_claim to Date column with formate (MM-dd-yyyy)
###Ensure  claim amount is >0

In [0]:
df = spark.sql("""
    SELECT * FROM (
        SELECT c.claim_id,c.claim_amount,c.LastUpdatedTimeStamp,c.policy_id,c.claim_status,to_date(date_format(c.date_of_claim, 'MM-dd-yyyy'),'MM-dd-yyyy') as date_of_claim,
               ROW_NUMBER() OVER (PARTITION BY c.claim_id ORDER BY c.LastUpdatedTimeStamp DESC) as rn
        FROM bronzelayer.Claim c
        INNER JOIN bronzelayer.Policy p ON c.policy_id = p.policy_id
        WHERE c.claim_id IS NOT NULL 
        AND c.policy_id IS NOT NULL 
        AND c.claim_status IS NOT NULL 
        AND c.claim_amount IS NOT NULL 
        AND c.LastUpdatedTimeStamp IS NOT NULL
        and c.claim_amount >0
    ) WHERE rn = 1
""")
display(df)

In [0]:
df.createOrReplaceTempView("clean_claim")

In [0]:
 %sql
 MERGE INTO silverlayer.Claim AS T USING clean_claim AS S ON 
  t.claim_id = s.claim_id WHEN MATCHED THEN
  UPDATE SET

    t.claim_amount = s.claim_amount,
    t.LastUpdatedTimeStamp = s.LastUpdatedTimeStamp,
    t.policy_id = s.policy_id,
    t.claim_status = s.claim_status,
    t.date_of_claim = s.date_of_claim,
    t.merged_timestamp = current_timestamp()
   WHEN not MATCHED THEN 
   INSERT (claim_id,claim_amount, LastUpdatedTimeStamp, policy_id, claim_status, date_of_claim,merged_timestamp ) 
   VALUES (s.claim_id, s.claim_amount, s.LastUpdatedTimeStamp, s.policy_id, s.claim_status, s.date_of_claim,current_timestamp())

In [0]:
spark.sql(" UPDATE bronzelayer.claim set merge_flag = TRUE where merge_flag = FALSE")